# Credit Card Default Prediction with UCI_Credit_Card Dataset

This notebook builds and evaluates a machine learning model to predict credit card default using the UCI_Credit_Card dataset. The provided `train_data` and `test_data` folders are used for training and evaluation.


## 1. Import Required Libraries

Import pandas, numpy, scikit-learn, and any other necessary libraries for data handling, preprocessing, modeling, and evaluation.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import os

## 2. Load and Explore Training and Test Data

Load the training and test datasets from the provided folders. Display the first few rows and basic statistics to understand the data.

In [ ]:
# Define paths to train and test data folders
train_folder = './data/train_data'
test_folder = './data/test_data'

# Find CSV files in the folders
train_files = [os.path.join(train_folder, f) for f in os.listdir(train_folder) if f.endswith('.csv')]
test_files = [os.path.join(test_folder, f) for f in os.listdir(test_folder) if f.endswith('.csv')]

# Load and concatenate all train and test files
train_df = pd.concat([pd.read_csv(f) for f in train_files], ignore_index=True) if train_files else pd.DataFrame()
test_df = pd.concat([pd.read_csv(f) for f in test_files], ignore_index=True) if test_files else pd.DataFrame()

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
display(train_df.head())
display(test_df.head())
print(train_df.describe())

## 3. Preprocess Data (Handle Missing Values, Encode Categoricals, Feature Scaling)

Check for and handle missing values, encode categorical variables if necessary, and apply feature scaling to numerical features.

In [ ]:
# Handle missing values
train_df = train_df.dropna()
test_df = test_df.dropna()

# Encode categorical variables if needed (assuming all are numeric except 'SEX', 'EDUCATION', 'MARRIAGE')
categorical_cols = ['SEX', 'EDUCATION', 'MARRIAGE']
for col in categorical_cols:
    if col in train_df.columns:
        le = LabelEncoder()
        train_df[col] = le.fit_transform(train_df[col])
        if col in test_df.columns:
            test_df[col] = le.transform(test_df[col])

# Feature scaling (excluding target and ID)
feature_cols = [col for col in train_df.columns if col not in ['ID', 'default.payment.next.month']]
scaler = StandardScaler()
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
if not test_df.empty:
    test_df[feature_cols] = scaler.transform(test_df[feature_cols])

## 4. Split Training Data into Features and Target

Separate the features (X) and the target variable (y = 'default.payment.next.month') from the training data.

In [ ]:
# Split features and target
X = train_df.drop(['ID', 'default.payment.next.month'], axis=1)
y = train_df['default.payment.next.month']

## 5. Train a Classification Model

Train a classification model (Random Forest) using the preprocessed training data.

In [ ]:
# Train Random Forest Classifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

## 6. Evaluate Model on Training Data

Evaluate the trained model using metrics such as accuracy, precision, recall, F1-score, and confusion matrix on the training set.

In [ ]:
# Predict on training data
y_pred_train = rf.predict(X)

print('Accuracy:', accuracy_score(y, y_pred_train))
print('Precision:', precision_score(y, y_pred_train, average='weighted'))
print('Recall:', recall_score(y, y_pred_train, average='weighted'))
print('F1 Score:', f1_score(y, y_pred_train, average='weighted'))
print('\nClassification Report:\n', classification_report(y, y_pred_train))
print('Confusion Matrix:\n', confusion_matrix(y, y_pred_train))

## 7. Make Predictions on Test Data

Apply the trained model to the preprocessed test data to generate predictions for credit card default.

In [ ]:
# Make predictions on test data (if available)
if not test_df.empty:
    X_test = test_df.drop(['ID'], axis=1) if 'ID' in test_df.columns else test_df
    test_preds = rf.predict(X_test)
    test_df['default.payment.next.month'] = test_preds
    print(test_df[['ID', 'default.payment.next.month']].head())
else:
    print('No test data available.')

## 8. Save Predictions to Output File

Save the test predictions to a CSV file for submission or further analysis.

In [ ]:
# Save predictions to CSV
if not test_df.empty:
    output = test_df[['ID', 'default.payment.next.month']] if 'ID' in test_df.columns else test_df
    output.to_csv('test_predictions.csv', index=False)
    print('Predictions saved to test_predictions.csv')

## 9. Train Second Model with Augmented (Synthetic) Data

Now, we will train a second model using the original training data combined with synthetic data. We will then predict on the same test set and compare both models' performance.

In [ ]:
# Load synthetic data (assumed to be in 'synthetic_rows.txt' as JSON)
import json
with open('synthetic_rows.txt', 'r') as f:
    synthetic_data = json.load(f)
synthetic_df = pd.DataFrame(synthetic_data)

# Preprocess synthetic data: encode categoricals and scale features
for col in categorical_cols:
    if col in synthetic_df.columns:
        le = LabelEncoder()
        synthetic_df[col] = le.fit_transform(synthetic_df[col])

synthetic_df[feature_cols] = scaler.transform(synthetic_df[feature_cols])

# Combine original and synthetic training data
augmented_train_df = pd.concat([train_df, synthetic_df], ignore_index=True)

# Split features and target for augmented data
X_aug = augmented_train_df.drop(['ID', 'default.payment.next.month'], axis=1)
y_aug = augmented_train_df['default.payment.next.month']

In [ ]:
# Train second Random Forest model on augmented data
rf_aug = RandomForestClassifier(n_estimators=100, random_state=42)
rf_aug.fit(X_aug, y_aug)

In [ ]:
# Evaluate second model on original training data
y_pred_train_aug = rf_aug.predict(X)
print('Augmented Model - Accuracy:', accuracy_score(y, y_pred_train_aug))
print('Augmented Model - Precision:', precision_score(y, y_pred_train_aug, average='weighted'))
print('Augmented Model - Recall:', recall_score(y, y_pred_train_aug, average='weighted'))
print('Augmented Model - F1 Score:', f1_score(y, y_pred_train_aug, average='weighted'))
print('\nAugmented Model - Classification Report:\n', classification_report(y, y_pred_train_aug))
print('Augmented Model - Confusion Matrix:\n', confusion_matrix(y, y_pred_train_aug))

In [ ]:
# Predict on test data with augmented model
if not test_df.empty:
    X_test = test_df.drop(['ID'], axis=1) if 'ID' in test_df.columns else test_df
    test_preds_aug = rf_aug.predict(X_test)
    test_df['default.payment.next.month_aug'] = test_preds_aug
    print(test_df[['ID', 'default.payment.next.month_aug']].head())
else:
    print('No test data available for augmented model.')

In [ ]:
# Save augmented model predictions to CSV
if not test_df.empty:
    output_aug = test_df[['ID', 'default.payment.next.month_aug']] if 'ID' in test_df.columns else test_df
    output_aug.to_csv('test_predictions_augmented.csv', index=False)
    print('Augmented model predictions saved to test_predictions_augmented.csv')

## 10. Compare Both Models

Compare the performance of the original and augmented models on the training data and show the difference in predictions on the test set.

In [ ]:
# Compare predictions on test set (if available)
if not test_df.empty and 'default.payment.next.month' in test_df.columns and 'default.payment.next.month_aug' in test_df.columns:
    comparison = test_df[['ID', 'default.payment.next.month', 'default.payment.next.month_aug']] if 'ID' in test_df.columns else test_df[['default.payment.next.month', 'default.payment.next.month_aug']]
    comparison['same_prediction'] = comparison['default.payment.next.month'] == comparison['default.payment.next.month_aug']
    print(comparison.head())
    print(f"Number of same predictions: {comparison['same_prediction'].sum()} / {len(comparison)}")
    print(f"Percentage same: {100 * comparison['same_prediction'].mean():.2f}%")
else:
    print('Test predictions for both models not available for comparison.')